# M3 · Evaluación con RAGAS y Cruce con el Harness de M2


1. **Faithfulness** (Fidelidad de la respuesta generada al contexto clínico).
2. **Context Precision** (Precisión de los fragmentos recuperados por el retrieval).
3. **Context Recall** (Cobertura de la evidencia necesaria para la recomendación clínica).
4. **Answer Relevancy** (Pertinencia de la respuesta frente a la pregunta original).
5. **Cruce con el Harness de M2** (F1 de extracción de Clinical BERT por documento vs RAG) y diagnóstico automático de fallas.


## 1 · Montar Google Drive (Datos Históricos de M2)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/TopicosIA/Proyecto-Salud')
print('PROJECT_ROOT:', PROJECT_ROOT)

## 2 · Clonar el Repositorio

In [ ]:
# Regresar a la raiz de Colab antes de borrar para mantener getcwd valido
%cd /content

!rm -rf /content/Topicos-IA
!git clone --branch dev --single-branch https://github.com/luisNP21/Topicos-IA.git /content/Topicos-IA

REPO_DIR = Path('/content/Topicos-IA')
%cd {REPO_DIR}/M3/ragas

# Verificamos si existe el archivo de M2 para el cruce (en Drive o en el repo clonado)
dim1_drive = PROJECT_ROOT / 'M2/ejecucion/outputs_gold/resultado_dimension1.json'
dim1_repo  = REPO_DIR / 'M2/ejecucion/outputs_gold/resultado_dimension1.json'

if dim1_drive.exists():
    print('Resultado M2 encontrado en Google Drive.')
elif dim1_repo.exists():
    print('Resultado M2 encontrado en el repositorio clonado.')
else:
    print('Aviso: resultado_dimension1.json no encontrado (se usara respaldo integrado).')


## 3 · Instalación de Dependencias

In [ ]:
%pip install -q -r requirements.txt
print('Dependencias de RAGAS instaladas.')

## 4 · Configuración de Secretos (.env)
Obtiene `GROQ_API_KEY` desde los Secretos de Colab (icono ).

In [ ]:
import os
from google.colab import userdata

try:
    groq_key = userdata.get('GROQ_API_KEY')
    os.environ['GROQ_API_KEY'] = groq_key
    print('GROQ_API_KEY configurada con éxito ')
except Exception:
    print('Aviso: GROQ_API_KEY no encontrada en Secretos. Para modo real configúrala en el icono .')

with open('/content/Topicos-IA/.env', 'w') as f:
    f.write(f'PROJECT_ROOT={PROJECT_ROOT}\n')
    if 'GROQ_API_KEY' in os.environ:
        f.write(f'GROQ_API_KEY={os.environ["GROQ_API_KEY"]}\n')

## 5 · Evaluación en Modo Mock (Rápido y Local)
Evalúa la tubería usando embeddings multilingües y genera la tabla de cruce con M2.

In [ ]:
import json
from pathlib import Path

# Comprobar si ya existen resultados generados previamente en Drive o en el repo
gen_out_drive = PROJECT_ROOT / 'M3/outputs/resultado_generacion.json'
gen_out_repo  = REPO_DIR / 'M3/outputs/resultado_generacion.json'

if not gen_out_drive.exists() and not gen_out_repo.exists():
    print('resultado_generacion.json no encontrado. Ejecutando modulo de generacion...')
    (REPO_DIR / 'M3/outputs').mkdir(parents=True, exist_ok=True)
    !PROJECT_ROOT=/content/Topicos-IA python /content/Topicos-IA/M3/generacion/run_generacion.py --config /content/Topicos-IA/M3/generacion/config.yaml

# Identificar la ruta activa del archivo generado
target_file = gen_out_drive if gen_out_drive.exists() else gen_out_repo

if target_file.exists():
    with open(target_file, 'r', encoding='utf-8') as f:
        gen_data = json.load(f)
    casos = gen_data.get('resultados', [])
    print(f'Archivo de generacion cargado desde: {target_file}')
    print(f'Total de casos clinicos generados: {len(casos)}')
    for i, c in enumerate(casos, 1):
        ent = c.get('entidad', '')
        n_ctx = len(c.get('contexts', []))
        ans = c.get('answer', '')
        print(f'  [{i}] Entidad: {ent} | Contextos: {n_ctx} | Longitud respuesta: {len(ans)} chars')
else:
    print('Aviso: No se encontro resultado_generacion.json. La evaluacion usara el set simulado integrado.')


## 6 · Evaluación en Modo Real (RAGAS Oficial + Juez Groq)
Corre la librería oficial `ragas` utilizando el modelo `qwen/qwen3.8-27b`.

In [ ]:
!python run_ragas.py --config config.yaml --modo real